**Loading Dependencies**

In [ ]:
pip install sentence_transformers huggingface langchain langchain_community pinecone pinecone-client pinecone-notebooks pypdf langchain-huggingface chromadb rank_bm25 langchain-groq pinecone-text langchain_pinecone

In [ ]:
pip install sentence_transformers

**Loading Sentence Transformer for Embeddings**

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

**Loading Documents using PyPDFLoader**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
docpath = "/content/drive/MyDrive/VectorSave/cardiology/cardiology-explained.pdf"
loader = PyPDFLoader(docpath)

In [ ]:
docs = loader.load()

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores.chroma import Chroma

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500 ,chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
len(chunks)

1060

**Using Chroma as Vectorstore**

In [ ]:
from langchain.vectorstores import Chroma
vectorstore = Chroma.from_documents(chunks, embeddings)

**Creating keyword based retriever using BM25**

**Creating dense vector retriever using vectorstore database**

In [ ]:
from langchain_community.retrievers import BM25Retriever
keyword_retriever = BM25Retriever.from_documents(chunks, k=3)     # BM25 will be the keyword retriver or sparse retriever
vectorstore_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})    # Dense Vector embeddings will be retrived using vectorstore retriever
                                                                            # k is set to 3 as it will retrieve top 3 documents

**Loading the ensemble retriever to perform hybrid search**

In [ ]:
from langchain.retrievers.ensemble import EnsembleRetriever         # Ensemble Retriver

ensemble_retriever = EnsembleRetriever(retrievers=[vectorstore_retriever,
                                                   keyword_retriever],
                                       weights=[0.7, 0.3])                # Alpha values or weights be assigned as 0.7 and 0.3 for this use-case

In [ ]:
import nltk         #installing more dependencies
nltk.download('punkt_tab')

**Re-Ranking using Cross Encoder**

***get_relevance_scores*** : the function will take the query and documents retrieved as input and create question-document pairs which will be scored by the cross encoder.
The function will return ***scores*** as the output.

The scores are logit scores which are un-normalized

In [ ]:
from sentence_transformers import CrossEncoder

def get_relevance_scores(question, documents):
    ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
    pairs = [[question, doc.page_content] for doc in documents]
    scores = ce.predict(pairs)
    return scores

**sort_and_display_documents** : this function will take the scores from the cross encoder along witht the retrieved documents and order them in descending order. This will help to retrieve top ranked documents.

As a secondary functionality it will also display the ordered re-ranked results along with their scores and other meta-data such as source, page number and content

In [ ]:
def sort_and_display_documents(scores, documents, display_limit=200):

    # Pair scores with documents
    scored_docs = [(score, doc) for score, doc in zip(scores, documents)]

    # Sort by scores in descending order
    sorted_docs = sorted(scored_docs, key=lambda x: x[0], reverse=True)

    # Extract the reordered documents
    reordered_docs = [doc for _, doc in sorted_docs]

    # Print scores, source, page numbers, document content
    print("Documents Sorted by Scores:\n")
    for score, doc in sorted_docs:
        # Extract file name from source path
        source_file_name = os.path.basename(doc.metadata.get('source', 'N/A'))

        print(f"Score: {score}, \nSource: {source_file_name}, Page: {doc.metadata.get('page', 'N/A')}, \nDoc:\n{doc.page_content[:display_limit]}\n\n")

    return reordered_docs

In [ ]:
from langchain_core.output_parsers import StrOutputParser

**get_highest_ranked_document** : This is an extension of sort_and_display_function. This will just return the highest ranked document based on the logit score.

This will be used further when we want the context to be minimal and restricted to only one document.

In [ ]:
def get_highest_ranked_document(scores, documents):

    if scores.size == 0 or not documents or len(scores) != len(documents):
        raise ValueError("Scores and documents must be non-empty and of the same length.")

    # Pair scores with documents and find the maximum by score
    highest_ranked = max(zip(scores, documents), key=lambda x: x[0])

    # Return the document with the highest score
    return highest_ranked[1]

**get_best_doc**: this is same as get_highest_ranked_document but instead of taking scores as the input we just provide the question and documents retrieved.

This is easier to call as a function.

In [ ]:
def get_best_doc(question, documents):
    scores = get_relevance_scores(question, documents)
    return get_highest_ranked_document(scores, documents)

**full_retrieval_time** : Function to map the retrieval time using ensemble retrieval and cross encoder re-ranking

In [ ]:
def full_retrieval(question):
    docs = ensemble_retriever.invoke(question)
    best_doc = get_best_doc(question, docs)
    return best_doc

**Loading the LLM using Groq**

In [ ]:
from langchain_groq import ChatGroq

groqllm=ChatGroq(groq_api_key=userdata.get('GROQ'),
                 model_name= "groq/compound")

We will be creating a prompt template to instruct the LLM to generate additional queries based on the user query. The queries will such thst they expand the topic related to the user query and not just output 4 similar ones.




In [ ]:
from langchain.prompts import ChatPromptTemplate

multi_template = """You are a helpful assistant that generates exactly 4 search queries expanding on a given topic.\n
    "Input question: {question}\n\n"
    "Output only a valid JSON array of 4 strings representing the search queries.\n"
    "Do not include any explanations, markdown, or additional text. Do not try to answer the question. only the raw JSON array."""

multi_prompt = ChatPromptTemplate.from_template(multi_template)

In [ ]:
generate_queries = ( multi_prompt | groqllm | StrOutputParser() | (lambda x: x.split("\n")))

In [ ]:
multi_query1 = "what are the classic symptoms of angina pectoris?"

In [ ]:
multi_questions1 = generate_queries.invoke({"question":multi_query1})

In [ ]:
multi_questions1

['["what are the common signs of angina pectoris","classic symptoms of stable angina pectoris","symptoms of unstable angina pectoris","how to identify angina pectoris symptoms"]']

In [ ]:
multi_questions2 = generate_queries.invoke({"question":"what is a heart murmur?"})

In [ ]:
multi_questions2

['["what are the causes of a heart murmur","heart murmur symptoms and treatment","types of heart murmurs in adults","can a heart murmur be normal"]']

Parser to help format output in form of 4 separate sub-questions

In [ ]:
import json
import re

def parse_llm_queries(output):
    """
    Extracts up to 4 search queries from an LLM output.
    Handles cases where:
      - output is a list of strings or a single string
      - JSON array is wrapped in quotes (e.g. '["..."]')
      - JSON code blocks are used
    Returns: list of 4 query strings
    """

    # Step 1: Normalize input
    if isinstance(output, list):
        text = "\n".join(output)
    elif isinstance(output, str):
        text = output
    else:
        raise TypeError("Input must be a string or list of strings.")

    text = text.strip()

    # Step 2: Extract JSON array (with or without code fences)
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if not match:
        raise ValueError("No JSON array found in the output.")

    json_content = match.group(0).strip()

    # Step 3: Parse JSON safely (handle double-encoded cases)
    try:
        data = json.loads(json_content)
        if isinstance(data, str):  # JSON string containing JSON array
            data = json.loads(data)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON content: {e}")

    # Step 4: Validate list structure
    if not isinstance(data, list) or not all(isinstance(q, str) for q in data):
        raise ValueError("Parsed content is not a list of strings.")

    # Step 5: Ensure exactly 4 queries
    return data[:4]


In [ ]:
sub_queries = parse_llm_queries(multi_questions2)
print(sub_queries)

['what are the causes of a heart murmur', 'heart murmur symptoms and treatment', 'types of heart murmurs in adults', 'can a heart murmur be normal']


Prompt Template for Answering based on context

In [ ]:
from langchain_core.prompts import PromptTemplate
gqtemplate = """Answer the question :
{question}
Based on the following context :
{context}

"""
gqprompt = PromptTemplate(
    template=gqtemplate,
    input_variables=["context", "question"],  # The context will be the top retrieved document and the question is the user query

)

In [ ]:
groqchain = gqprompt | groqllm | StrOutputParser()

**groq_test** : defining a function that will take user query as input and invoke the RAG chain. The context will invoke the cache_retrieval_function which will return the best ranked document. This document will now be passed as context alongside the user query into the prompt template named gq_template for the LLM Gemma to answer the question using Retrieved context from the textbook

In [ ]:
def groq_test(question):
  result = groqchain.invoke({"context": full_retrieval(question), "question": question})
  return result

Creating a function answer_all which will call the multi_questions and cleaned_questions function to get the additional queries.

A list rag_answers will be used to store the answers of the additional queries.

For every additional query we will invoke the rag chain to get a response from the LLM. These answers will be stored in rag_answers

The function will output the answers along with the additional questions.

In [ ]:
def answer_all(query):
  multi_questions = generate_queries.invoke({"question":query})
  cleaned_questions = parse_llm_queries(multi_questions)
  rag_answers= []
  for q in cleaned_questions:
    answer = groqchain.invoke({"context": full_retrieval(q), "question": q})
    rag_answers.append(answer)
  return query,rag_answers, cleaned_questions

In [ ]:
query1,rag_answers1, cleaned_questions1 = answer_all(multi_query1)

In [ ]:
query1

'what are the classic symptoms of angina pectoris?'

In [ ]:
rag_answers1

['**Answer – Common signs (symptoms) of angina pectoris**\n\nThe source you gave (a short excerpt from a cardiology textbook) mentions the term “angina pectoris” but does not list its clinical manifestations.\u202fTo answer the question we rely on standard medical descriptions of angina, which are consistent across reputable sources (Mayo Clinic, Healthdirect, Penn Medicine, etc.).\n\n| Symptom | Typical description |\n|---------|----------------------|\n| **Chest discomfort/pain** | A pressure, heaviness, tightness, squeezing, burning or “crushing” sensation under the breastbone. Often triggered by exertion or emotional stress and relieved by rest or nitroglycerin. |\n| **Radiating pain** | May spread to the **arms (usually left), shoulders, back, neck, jaw, or even the ears**. |\n| **Shortness of breath (dyspnea)** | Often accompanies the chest pain, especially during activity. |\n| **Fatigue / weakness** | A feeling of being unusually tired or weak, sometimes described as “extreme t

In [ ]:
cleaned_questions1

['what are the common signs of angina pectoris',
 'classic symptoms of stable angina pectoris',
 'symptoms of unstable angina pectoris',
 'how to identify angina pectoris symptoms']

In [ ]:
query2,rag_answers2, cleaned_questions2 = answer_all("what is the most common cause of ischemic heart disease?")

In [ ]:
cleaned_questions2

['what are the risk factors for ischemic heart disease',
 'most common cause of ischemic heart disease',
 'how does atherosclerosis cause ischemic heart disease',
 'is high blood pressure a cause of ischemic heart disease']